In [13]:
import pandas as pd
import numpy as np
import tensorflow as tf
from keras.saving import load_model
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [6]:
SEED = 42
DATA_PATH = 'C:\\Users\\meerh\\OneDrive\\Documentos\\Projetos\\esp32-fall-detection\\data\\processed\\acc_gyr.csv'
MODEL_PATH = 'C:\\Users\\meerh\\OneDrive\\Documentos\\Projetos\\esp32-fall-detection\\models\\best_model_exp1.keras'

In [7]:
model = load_model(MODEL_PATH)

In [4]:
df = pd.read_csv(DATA_PATH)
df.shape

X = df.drop('label', axis=1)
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=SEED, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, test_size=0.5, random_state=SEED, stratify=y_test)

X_train.shape, y_train.shape, X_test.shape, y_test.shape, X_val.shape, y_val.shape

((67760, 6), (67760,), (14520, 6), (14520,), (14520, 6), (14520,))

# Quantização do Modelo

In [10]:
def representative_dataset():
    # Itera sobre as amostras selecionadas
    for i in range(len(X_test)):
        amostra = X_test.iloc[i].to_numpy(dtype=np.float32)
        yield [np.expand_dims(amostra, axis=0)]

In [11]:
# 3. Inicializar e configurar o conversor
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Habilitar a otimização de tamanho (gatilho para a quantização)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Injetar a função com os dados de calibração
converter.representative_dataset = representative_dataset

# Forçar o TensorFlow a converter TODAS as operações suportadas para inteiros
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]

# Alterar os pinos de entrada e saída do modelo de float32 para int8
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

In [12]:
tflite_model = converter.convert()

INFO:tensorflow:Assets written to: C:\Users\meerh\AppData\Local\Temp\tmp428hlue8\assets


INFO:tensorflow:Assets written to: C:\Users\meerh\AppData\Local\Temp\tmp428hlue8\assets


Saved artifact at 'C:\Users\meerh\AppData\Local\Temp\tmp428hlue8'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 6), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  2355738394256: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2355738394064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2353642160976: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2353642161552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2353642161168: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2353642162896: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2353642161936: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2353642162320: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2353642162512: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2353642162704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2353642160784: T

c:\Users\meerh\Documents\esp32-fall-detection\venv\Lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


In [14]:
caminho_tflite = "C:\\Users\\meerh\\Documents\\esp32-fall-detection\\models\\model_quantized.tflite"
with open(caminho_tflite, "wb") as f:
    f.write(tflite_model)